# Decomposed Noisy FSim Ansatz

Build the Cirq chemistry ansatz, decompose every restricted `FSim` gate into `CZ + 1Q` gates, apply a location-aware empirical noise model, and measure the Hamiltonian from the noisy density matrix trace.

In [3]:
from __future__ import annotations

from pathlib import Path

import cirq
import numpy as np
from cirq.contrib.svg import SVGCircuit

from main_cursor_lib import (
    DEFAULT_AMP_DAMP_GAMMA,
    DEFAULT_DEPOL_PROB,
    DEFAULT_HIGH_CZ_MULTIPLIER,
    DEFAULT_LEAKAGE_APPROX_PROB,
    DEFAULT_PHASE_DAMP_GAMMA,
    LocationAwareDecomposedNoise,
    load_hamiltonian_paths,
    load_observable_h,
    ordered_parameter_symbols,
    params_per_layer,
    prepare_decomposed_ansatz_cirq,
    trace_energy,
)
from shot_measurement import (
    estimate_energy_from_noisy_rho_shots,
    run_mitigation,
)

# -------------------------------------------------------------------
# Configuration (3 layers = 21 parameters for H4 / num_spatial=4)
# -------------------------------------------------------------------
H_atom = 4
bond_length = 2.0
num_spatial = 4
ansatz_layers = 3

# Layer-3 baseline vector (same as docs/scripts/layer3_mitigation_sweep.py).
vqe_parameters = np.array([
    1.02328779,
    1.33825349,
    0.78981901,
    0.36255437,
    0.64652655,
    3.31926241,
    -0.07988963,
    0.22301099,
    0.4992053,
    1.3831351,
    5.50652141,
    2.7144864,
    2.14070825,
    -0.10898955,
    -0.2443687,
    -0.45224915,
    0.95313939,
    -1.18188991,
    -1.73092033,
    1.36303661,
    1.41789702,
])

# Finite-shot measurement options.
num_shots = 8192
measurement_scheme = "ogm"  # ogm|adaptive_shadows|derandomization|shadow_grouping_bernstein|shadow_grouping_inconfidence|direct_pauli
sampling_seed = 1234

# Readout calibration (top-level arguments).
p_0_success = np.array([0.93, 0.87, 0.87, 0.88, 0.90, 0.89, 0.87, 0.91])
p_1_success = np.array([0.95, 0.91, 0.95, 0.96, 0.94, 0.94, 0.92, 0.95])

apply_readout_noise = True
apply_rem = True

# Shadowgrouping integration config.
SHADOWGROUPING_ROOT = "/Users/zacharyhe/shadowgrouping"
ogm_file = f"{SHADOWGROUPING_ROOT}/haozhaowu/H4/hamil_class/ogm_outputs/OGM_H4_bond_{bond_length:.1f}.txt"

# Noise parameters (Weber/Sycamore-motivated defaults in main_cursor_lib).
random_seed = 1234
theta_std_dev = 0.026 * 2
phi_std_dev = 0.037 * 2
amp_damp_gamma = DEFAULT_AMP_DAMP_GAMMA
phase_damp_gamma = DEFAULT_PHASE_DAMP_GAMMA
depol_prob = DEFAULT_DEPOL_PROB
high_cz_multiplier = DEFAULT_HIGH_CZ_MULTIPLIER
leakage_approx_prob = DEFAULT_LEAKAGE_APPROX_PROB

# -------------------------------------------------------------------
# Mitigation mode (single switch).
# -------------------------------------------------------------------
# One of: "none" | "zne" | "cdr" | "both".
mitigation_mode = "both"

# ZNE sub-config (used by mode in {"zne", "both"}).
zne_scales = [1.0, 2.0, 3.0]
zne_fit_order = 1

# CDR sub-config (used by mode in {"cdr", "both"}).
cdr_num_circuits = 24
cdr_t_max = 32              # hard non-Clifford gate budget per training circuit
cdr_min_snap_fraction = 0.0  # optional diversity knob
cdr_seed = 7

print_circuit = True
workspace = Path.cwd()

ppl = params_per_layer(num_spatial)
expected_num_parameters = ansatz_layers * ppl
if len(vqe_parameters) != expected_num_parameters:
    raise ValueError(
        f"ansatz_layers={ansatz_layers} requires {expected_num_parameters} parameters "
        f"({ppl} per layer), got {len(vqe_parameters)}."
    )
if len(p_0_success) != 2 * num_spatial or len(p_1_success) != 2 * num_spatial:
    raise ValueError("Readout calibration arrays must have length 2 * num_spatial.")

print(f"Workspace: {workspace}")
print(f"Using {ansatz_layers} layers and {len(vqe_parameters)} parameters.")
print(f"Measurement scheme: {measurement_scheme} | shots={num_shots} | sampling_seed={sampling_seed}")
print(f"Mitigation mode: {mitigation_mode}")


Workspace: /Users/zacharyhe/cross_chips_sim
Using 3 layers and 21 parameters.
Measurement scheme: ogm | shots=8192 | sampling_seed=1234
Mitigation mode: both


In [4]:
# Build decomposed ansatz and Hamiltonian.
ansatz_circuit, ansatz_qubits = prepare_decomposed_ansatz_cirq(num_spatial, ansatz_layers)
observable_H = load_observable_h(workspace, ansatz_qubits, H_atom, bond_length)
_, hamiltonian_pkl_path, hamiltonian_text_path = load_hamiltonian_paths(workspace, H_atom, bond_length)
print(f"Hamiltonian candidates: {hamiltonian_pkl_path.name}, {hamiltonian_text_path.name}")

symbols = ordered_parameter_symbols(num_spatial, ansatz_layers)
resolver = cirq.ParamResolver(dict(zip(symbols, vqe_parameters)))
hamiltonian_matrix = observable_H.matrix(qubits=ansatz_qubits)
print(f"Number of parameters: {len(symbols)}")


Hamiltonian candidates: H4_bond_2.pkl, H4_bond_2_pauli_convention.txt
Number of parameters: 21


In [5]:
# Draw the decomposed circuit for visual inspection.
if print_circuit:
    SVGCircuit(ansatz_circuit)


In [6]:
# Apply location-aware noise to the decomposed circuit.
chem_noise_model = LocationAwareDecomposedNoise(
    amp_damp_gamma=amp_damp_gamma,
    phase_damp_gamma=phase_damp_gamma,
    depol_prob=depol_prob,
    high_cz_multiplier=high_cz_multiplier,
    leakage_approx_prob=leakage_approx_prob,
)
noisy_ansatz_circuit = ansatz_circuit.with_noise(chem_noise_model)
if print_circuit:
    print(f"Noisy circuit depth: {len(noisy_ansatz_circuit)} moments")


Noisy circuit depth: 220 moments


In [7]:
# Noisy density-matrix trace energy and shot-based energies.
noisy_simulator = cirq.DensityMatrixSimulator(seed=random_seed)
resolved_noisy_circuit = cirq.resolve_parameters(noisy_ansatz_circuit, resolver)
noisy_result = noisy_simulator.simulate(resolved_noisy_circuit, qubit_order=ansatz_qubits)
rho_noisy = noisy_result.final_density_matrix

trace_noisy_energy = trace_energy(hamiltonian_matrix, rho_noisy)
print(f"VQE parameters  Tr[H rho_noisy]: {trace_noisy_energy:.12f}")  # noisy trace

shot_est = estimate_energy_from_noisy_rho_shots(
    rho_noisy,
    observable_H,
    ansatz_qubits,
    num_shots=num_shots,
    measurement_scheme=measurement_scheme,
    p_0_success=p_0_success,
    p_1_success=p_1_success,
    apply_rem=apply_rem,
    apply_readout_noise=apply_readout_noise,
    sampling_seed=sampling_seed,
    ogm_file=ogm_file,
    shadowgrouping_root=SHADOWGROUPING_ROOT,
)
print(f"Shot estimate (unmitigated): {shot_est['energy_unmitigated']:.12f}")
print(f"Shot estimate (REM):         {shot_est['energy_rem']:.12f}")


VQE parameters  Tr[H rho_noisy]: -2.437268491136
Shot estimate (unmitigated): -2.372462663195
Shot estimate (REM):         -2.394257766707


In [8]:
# Exact noiseless energy for the fixed VQE parameters.
exact_simulator = cirq.Simulator(seed=random_seed)
resolved_exact_circuit = cirq.resolve_parameters(ansatz_circuit, resolver)
exact_state = exact_simulator.simulate(resolved_exact_circuit, qubit_order=ansatz_qubits).final_state_vector
exact_vqe_energy = np.vdot(exact_state, hamiltonian_matrix @ exact_state).real
print(f"VQE parameters  Exact noiseless energy: {exact_vqe_energy:.12f}")


VQE parameters  Exact noiseless energy: -2.971707567945


In [9]:
# Mitigation: dispatch on `mitigation_mode`.
base_noise_cfg = dict(
    amp_damp_gamma=amp_damp_gamma,
    phase_damp_gamma=phase_damp_gamma,
    depol_prob=depol_prob,
    leakage_approx_prob=leakage_approx_prob,
    high_cz_multiplier=high_cz_multiplier,
)
shot_cfg = dict(
    num_shots=num_shots,
    measurement_scheme=measurement_scheme,
    apply_readout_noise=apply_readout_noise,
    sampling_seed=sampling_seed,
    epsilon=0.1,
    ogm_file=ogm_file,
    shadowgrouping_root=SHADOWGROUPING_ROOT,
)
readout_cal = dict(p_0_success=p_0_success, p_1_success=p_1_success)
zne_cfg = dict(noise_scales=zne_scales, fit_order=zne_fit_order)
cdr_cfg = dict(
    num_circuits=cdr_num_circuits,
    t_max=cdr_t_max,
    min_snap_fraction=cdr_min_snap_fraction,
    seed=cdr_seed,
)

mit_out = run_mitigation(
    mitigation_mode,
    ansatz_circuit=ansatz_circuit,
    observable_h=observable_H,
    qubits=ansatz_qubits,
    target_resolver=resolver,
    target_params={s: float(v) for s, v in zip(symbols, vqe_parameters)},
    symbols=symbols,
    base_noise_cfg=base_noise_cfg,
    shot_cfg=shot_cfg,
    readout_cal=readout_cal,
    zne_cfg=zne_cfg,
    cdr_cfg=cdr_cfg,
    simulator_seed=random_seed,
)

print(f"-- Mitigation mode: {mit_out['mode']} --")
print(f"Target shot unmit: {mit_out['unmit_target']:.12f}")
print(f"Target shot REM:   {mit_out['rem_target']:.12f}")

if "trace_zne" in mit_out:
    tz = mit_out["trace_zne"]
    print("\nTrace-based ZNE points:")
    for s, e in zip(tz["noise_scales"], tz["trace_energies"]):
        print(f"  scale={s:.1f} -> Tr[H rho_noisy]={e:.12f}")
    print(f"Trace ZNE extrapolated:        {tz['energy_zne']:.12f}")

if "shot_zne" in mit_out:
    sz = mit_out["shot_zne"]
    print("\nShot-based ZNE points (unmitigated):")
    for s, e in zip(sz["noise_scales"], sz["shot_unmitigated_energies"]):
        print(f"  scale={s:.1f} -> E_shot={e:.12f}")
    print(f"Shot ZNE extrapolated (unmit): {sz['zne_unmitigated']:.12f}")
    if "zne_rem" in sz:
        print(f"Shot ZNE extrapolated (REM):   {sz['zne_rem']:.12f}")

if "cdr_models" in mit_out:
    models = mit_out["cdr_models"]
    fs = models.get("fit_scope", "total_energy")
    print(f"\nCDR fit_scope (dispatched): {mit_out.get('cdr_fit_scope', fs)}")
    t_rem = models["training_t_remaining"]
    if fs == "per_pauli":
        ntm = len(models["coeffs_unmit_to_exact_per_term"])
        print(f"CDR per-Pauli: {ntm} affine models (unmit/REM per Hamiltonian term)")
    else:
        a_b, b_b = models["coeffs_unmit_to_exact"]
        a_c, b_c = models["coeffs_rem_to_exact"]
        print(f"\nCDR Model B (unmit -> exact): a={a_b:.6f}, b={b_b:.6f}")
        print(f"CDR Model C (REM   -> exact): a={a_c:.6f}, b={b_c:.6f}")
    print(
        f"CDR training t_remaining: min={min(t_rem)}, "
        f"median={int(np.median(t_rem))}, max={max(t_rem)} "
        f"(t_max budget = {cdr_t_max})"
    )
    print(f"CDR-unmit corrected: {mit_out['cdr_unmit_corrected']:.12f}")
    print(f"CDR-REM   corrected: {mit_out['cdr_rem_corrected']:.12f}")

if "zne_of_cdr_unmit_target" in mit_out:
    print("\nSequential CDR-then-ZNE per-scale energies:")
    for s, eu, er in zip(
        mit_out["zne_scales"],
        mit_out["per_scale_cdr_unmit"],
        mit_out["per_scale_cdr_rem"],
    ):
        print(f"  scale={s:.1f} -> CDR-unmit={eu:.12f}, CDR-REM={er:.12f}")
    print(f"ZNE-of-CDR (unmit) -> s=0:     {mit_out['zne_of_cdr_unmit_target']:.12f}")
    print(f"ZNE-of-CDR (REM)   -> s=0:     {mit_out['zne_of_cdr_rem_target']:.12f}")

print(f"\nExact noiseless reference:     {exact_vqe_energy:.12f}")


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/zacharyhe/anaconda3/envs/cross_chips_sim/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/d0/x90641td65g9zg_8chvnbtcm0000gn/T/ipykernel_36144/3844943925.py", line 27, in <module>
    mit_out = run_mitigation(
              ^^^^^^^^^^^^^^^
  File "/Users/zacharyhe/cross_chips_sim/shot_measurement.py", line 1121, in run_mitigation
    models_at_scale, corrected = _train_and_apply_cdr(scaled_noise)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zacharyhe/cross_chips_sim/shot_measurement.py", line 1007, in _train_and_apply_cdr
    m = train_cf_models_per_pauli(
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zacharyhe/cross_chips_sim/shot_measurement.py", line 704, in train_cf_models_per_pauli
    rho = _simulate_noisy_rho_for_resolver(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Use

In [ ]:
# Head-to-head: default per-Pauli CDR vs legacy total-energy CDR (same configuration, mode `cdr` only).
cdr_cfg_base = dict(
    num_circuits=cdr_num_circuits,
    t_max=cdr_t_max,
    min_snap_fraction=cdr_min_snap_fraction,
    seed=cdr_seed,
)
mit_pp = run_mitigation(
    "cdr",
    ansatz_circuit=ansatz_circuit,
    observable_h=observable_H,
    qubits=ansatz_qubits,
    target_resolver=resolver,
    target_params={s: float(v) for s, v in zip(symbols, vqe_parameters)},
    symbols=symbols,
    base_noise_cfg=base_noise_cfg,
    shot_cfg=shot_cfg,
    readout_cal=readout_cal,
    cdr_cfg={**cdr_cfg_base},
    simulator_seed=random_seed,
)
mit_te = run_mitigation(
    "cdr",
    ansatz_circuit=ansatz_circuit,
    observable_h=observable_H,
    qubits=ansatz_qubits,
    target_resolver=resolver,
    target_params={s: float(v) for s, v in zip(symbols, vqe_parameters)},
    symbols=symbols,
    base_noise_cfg=base_noise_cfg,
    shot_cfg=shot_cfg,
    readout_cal=readout_cal,
    cdr_cfg={**cdr_cfg_base, "cdr_fit_scope": "total_energy"},
    simulator_seed=random_seed,
)
eref = float(exact_vqe_energy)
print("=== CDR at scale 1.0: |error| vs exact noiseless energy ===")
for name, mo in [("per_pauli (default)", mit_pp), ("total_energy (legacy)", mit_te)]:
    eu = abs(float(mo["cdr_unmit_corrected"]) - eref)
    er = abs(float(mo["cdr_rem_corrected"]) - eref)
    print(f"{name}: |err| unmit-corrected={eu:.6f} Ha  |  |err| REM-corrected={er:.6f} Ha")


NameError: name 'cdr_num_circuits' is not defined